# V-BAR quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamilhammoudeh/vbar/blob/main/examples/quickstart.ipynb)

This notebook installs V-BAR from PyPI, flies the PD baseline to a hold point behind a target,
prints the run's metrics, and rebuilds one small results table with the entry point of the command
that rebuilds the published ones.

In [ ]:
%pip install -q vbar

## One episode

The chaser starts 400 m behind the target on a circular 500 km orbit and has to stop at the hold
point 100 m behind it, outside a 50 m keep-out sphere. `record_episode` flies the run and
`evaluate` scores it from the true trajectory.

In [ ]:
from vbar import ApproachConfig, ApproachEnv
from vbar.agents import PDAgent
from vbar.metrics import evaluate, record_episode

env = ApproachEnv(ApproachConfig(start_state=(0, -400, 0, 0, 0, 0)))
metrics = evaluate(record_episode(env, PDAgent()))

for name in ("success", "time_to_complete", "delta_v", "terminal_pos_error", "min_margin"):
    print(f"{name}: {metrics[name]}")

## One table

A published table is a suite, an agent and the files its results and report go to.
`python -m vbar.eval reproduce reproduce.toml` rebuilds every table of the repository's manifest;
its suites hold 400 episodes or more. Here a manifest lists one table of two episodes. The
command line knows only the built-in suites, so the suite is registered first and `main`, the
entry point of `python -m vbar.eval`, is called with the same arguments.

In [ ]:
from pathlib import Path

from vbar.eval.__main__ import main
from vbar.scenarios import Suite, SuiteEntry, register_suite

register_suite(Suite("quickstart", 1, (SuiteEntry("approach-rbar", "validation", 2),)))

Path("quickstart.toml").write_text(
    "[[table]]\n"
    'name = "quickstart"\n'
    'suite = "quickstart-v1"\n'
    'agent = "vbar.agents:PDAgent"\n'
    'results = "quickstart.jsonl"\n'
    'markdown = "quickstart.md"\n',
    encoding="utf-8",
)
main(["reproduce", "quickstart.toml"])
print(Path("quickstart.md").read_text(encoding="utf-8"))

Running the cell again flies nothing: the results file already holds both episodes. A suite is
reported in full or not at all, and more seeds give tighter intervals.

To rebuild the published tables, clone the repository and run
`python -m vbar.eval reproduce reproduce.toml`. The evaluation harness is described in
[docs/evaluation.md](https://github.com/jamilhammoudeh/vbar/blob/main/docs/evaluation.md).